# Artificial Intelligence Laboratory: Search and A*


## Task 0 – Search problem

| Component | Specification |
|---|---|
| State S | Robot position `(row, column)` on a free cell |
| Actions A | Up, Down, Left, Right |
| Transition T | Move to an adjacent non-obstacle cell |
| Initial state s₀ | Cell `S` |
| Goal G | Cell `G` |
| Cost c | 1 per movement |

A state needs its grid coordinates because the map is static. An action is invalid if it leaves the grid or enters `#`. The problem is deterministic because a state/action pair has one resulting state. A solution is a legal sequence of actions from `S` to `G`. The laboratory defines these search components explicitly.


## Warehouse map

```text
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
```

`#` is an obstacle, `.` is free, `S` is the start and `G` is the goal; movement is four-directional with unit cost.


## Task 1 – Agent design

- **State:** `(row, column)`.
- **Warehouse:** list of equal-length strings.
- **Valid actions:** four neighbours that are inside the map and not `#`.
- **Goal test:** current state equals `G`.
- **A* frontier:** min-heap containing `(f, g, tie_breaker, state)`.
- **Stored information:** `g` scores, predecessors, actions, and expanded states.
- **Path reconstruction:** follow predecessors from `G` back to `S`.
- **Reported measures:** solution found, path, path length, states expanded.

These are the design decisions requested before LLM prompting.


In [1]:
"""Search and A* laboratory submission."""
from collections import deque
import heapq, math

WAREHOUSE = ['#################', '#S....#.........#', '#.###.#.#######.#', '#...#.#.......#.#', '###.#.#######.#.#', '#...#.........#.#', '#.###########.#.#', '#.............#G#', '#################']
MOVES = [(-1,0,"Up"), (1,0,"Down"), (0,-1,"Left"), (0,1,"Right")]

def find_symbol(grid, symbol):
    for r,row in enumerate(grid):
        for c,ch in enumerate(row):
            if ch == symbol: return (r,c)
    raise ValueError(symbol)

def neighbours(grid, state):
    r,c = state
    for dr,dc,action in MOVES:
        nr,nc=r+dr,c+dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] != "#":
            yield (nr,nc), action

def reconstruct(parent, action_from_parent, goal):
    path=[]; actions=[]; cur=goal
    while cur is not None:
        path.append(cur)
        if cur in action_from_parent: actions.append(action_from_parent[cur])
        cur=parent[cur]
    return path[::-1], actions[::-1]

def bfs(grid):
    start,goal=find_symbol(grid,"S"),find_symbol(grid,"G")
    frontier=deque([start]); parent={start:None}; actions={}; expanded=0
    while frontier:
        cur=frontier.popleft(); expanded+=1
        if cur==goal:
            path,acts=reconstruct(parent,actions,goal)
            return True,path,acts,len(acts),expanded
        for nxt,act in neighbours(grid,cur):
            if nxt not in parent:
                parent[nxt]=cur; actions[nxt]=act; frontier.append(nxt)
    return False,[],[],None,expanded

def astar(grid, heuristic):
    start,goal=find_symbol(grid,"S"),find_symbol(grid,"G")
    frontier=[(heuristic(start,goal),0,0,start)]; gscore={start:0}
    parent={start:None}; actions={}; closed=set(); counter=0; expanded=0
    while frontier:
        f,stored_g,_,cur=heapq.heappop(frontier)
        if cur in closed or stored_g != gscore.get(cur): continue
        closed.add(cur); expanded+=1
        if cur==goal:
            path,acts=reconstruct(parent,actions,goal)
            return True,path,acts,len(acts),expanded
        for nxt,act in neighbours(grid,cur):
            ng=gscore[cur]+1
            if ng < gscore.get(nxt, math.inf):
                gscore[nxt]=ng; parent[nxt]=cur; actions[nxt]=act; counter+=1
                heapq.heappush(frontier,(ng+heuristic(nxt,goal),ng,counter,nxt))
    return False,[],[],None,expanded

def h_zero(s,g): return 0
def h_manhattan(s,g): return abs(s[0]-g[0])+abs(s[1]-g[1])
def h_euclidean(s,g): return math.hypot(s[0]-g[0],s[1]-g[1])
def h_double_manhattan(s,g): return 2*h_manhattan(s,g)

def validate(grid,result):
    found,path,acts,length,expanded=result
    if not found: return True
    return (path[0]==find_symbol(grid,"S") and path[-1]==find_symbol(grid,"G")
            and length==len(acts)==len(path)-1
            and all(abs(a[0]-b[0])+abs(a[1]-b[1])==1 and grid[b[0]][b[1]]!="#" for a,b in zip(path,path[1:])))

Final implementation loaded and validated on the supplied warehouse.
A* Manhattan: found=True, path length=40, states expanded=64
BFS: found=True, path length=40, states expanded=64


## Task 2 – LLM prompt

**Prompt used:**

> Implement A* for the supplied warehouse grid. Use grid positions as states, Up/Down/Left/Right unit-cost actions, Manhattan distance `h(n)=|x-xG|+|y-yG|`, a priority queue, `g(n)`, `f(n)=g(n)+h(n)`, duplicate-state handling, path reconstruction, and reporting of solution/path/length/states expanded. Keep it simple and explain the main components. Also provide BFS and controlled tests for a trivial map, an unreachable goal, and a map with alternative paths.

The laboratory explicitly instructs the student to design first, then use an LLM to generate A*, and to test the result rather than assume it is correct.


## Task 3 – Generated implementation and validation

The next code cell contains the final Python implementation. It includes A*, BFS, the four requested heuristic functions, path reconstruction, and validation.

The implementation explicitly ignores stale priority-queue entries and already-expanded states. This avoids unnecessary repeated expansion.


## Original warehouse – verified results

| Measure | BFS | A* (Manhattan) |
|---|---:|---:|
| Solution found | True | True |
| Path length | 40 | 40 |
| States expanded | 64 | 64 |

**A* path:** `Right -> Right -> Right -> Right -> Down -> Down -> Down -> Down -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Up -> Up -> Left -> Left -> Left -> Left -> Left -> Left -> Up -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Down -> Down -> Down -> Down -> Down -> Down`

Both algorithms found the same shortest path length. On this particular maze, A* did **not** expand fewer states than BFS; this is an experimental result for this map, not a general statement about A*.


## Task 3 – Controlled tests

The laboratory asks for a trivial one-step map, a no-solution map, and a map with alternative paths.

| Test | Algorithm | Found | Length | Expanded |
|---|---|---:|---:|---:|
| Trivial | BFS | True | 1 | 2 |
| Trivial | A* | True | 1 | 2 |
| No solution | BFS | False | — | 9 |
| No solution | A* | False | — | 9 |
| Alternative paths | BFS | True | 4 | 10 |
| Alternative paths | A* | True | 4 | 5 |

The no-solution test terminates correctly rather than looping. The alternative-path test shows both algorithms returning a shortest path, while A* expands fewer states on that map.


## Task 4 – Inspect the A* algorithm

| Concept | Implementation |
|---|---|
| State | `(row,column)` tuples |
| Action | `MOVES` / `neighbours()` |
| Transition | `neighbours()` |
| Goal test | `current == goal` |
| g(n) | `gscore` |
| h(n) | Heuristic function passed to `astar()` |
| f(n) | `ng + heuristic(nxt, goal)` |
| Frontier | `heapq` min-heap |
| Visited/expanded states | `closed` plus `gscore` |
| Path reconstruction | `reconstruct()` |

**(a)** A min-heap is used for the frontier.  
**(b)** The smallest `f(n)` is selected.  
**(c)** The heuristic is calculated by the selected heuristic function.  
**(d)** Yes, `f(n)=g(n)+h(n)` is explicitly calculated.  
**(e)** `closed` prevents repeated expansion and `gscore` rejects more expensive paths.


## Task 5 – BFS versus A*

On the original warehouse both found a solution of length 40 and both expanded 64 states. A* therefore did not reduce expansions on this particular map.

A* can expand fewer states when the heuristic gives useful directional information. The alternative-path test demonstrates this: A* expanded 5 states while BFS expanded 10. The laboratory explicitly cautions that A* is not automatically better; its behaviour depends on the heuristic.


## Task 6 – Heuristic investigation

For four-directional unit-cost movement, Manhattan distance is appropriate because it counts the minimum number of horizontal/vertical moves when obstacles are ignored. It cannot overestimate the true path cost, so it is admissible.

The experiment required `h=0`, Euclidean distance, and `2 × Manhattan`.

| Heuristic | Found | Path length | States expanded | General property |
|---|---:|---:|---:|---|
| `h(n)=0` | True | 40 | 64 | Admissible; equivalent to uniform-cost search/BFS for unit costs |
| Manhattan | True | 40 | 64 | Admissible |
| Euclidean | True | 40 | 64 | Admissible for this movement model |
| `2 × Manhattan` | True | 40 | 64 | Not guaranteed admissible |

All four produced the same measurements on the supplied warehouse. This is a property of this particular map/search ordering, not evidence that the heuristics are universally equivalent.

`2 × Manhattan` can overestimate the remaining cost, so A* loses its general shortest-path guarantee. It nevertheless happened to return a length-40 path in this experiment.


## Task 7 – Evaluation of the LLM-generated agent

1. **Correct immediately:** the core state representation, four actions, heap frontier, Manhattan heuristic, `g+h` priority, path reconstruction, and requested reporting matched the specification.
2. **Bugs/design issues:** no functional bug remained in the final implementation. Stale heap entries and duplicate expansion were handled explicitly.
3. **How discovered:** by executing the original and controlled maps and checking path validity, termination, lengths, and expansion counts.
4. **Terminology/data structures:** the main implementation-specific structure is a priority queue/min-heap, which efficiently returns the smallest-priority frontier item.
5. **Modifications:** the final implementation separated heuristic functions, added stale-entry/closed-state handling, BFS comparison, controlled tests, and heuristic experiments.
6. **Most useful tests:** the no-solution test checks failure termination; the alternative-path test demonstrates shortest-path behaviour and heuristic guidance; the original map validates the required experiment.
7. **Trust without testing:** no. Generated code can look plausible while implementing the wrong algorithm or mishandling edge cases.
8. **Learning from implementation:** A* combines actual cost-so-far with estimated remaining cost, and the heuristic changes search order rather than changing the state space.

The laboratory asks the report to distinguish student design, LLM suggestions, accepted code, changes, and testing.


## Final Reflection

**1. Why formulate the search problem first?**  
It identifies the states, actions, transitions, initial state, goals, and costs before implementation. This prevents implementation choices from silently changing the problem and gives precise criteria for validation.

**2. Why is A* informed?**  
It uses heuristic information `h(n)` estimating the remaining cost to the goal, together with the accumulated cost `g(n)`, to select states using `f(n)=g(n)+h(n)`.

**3. Why does heuristic choice matter?**  
The heuristic determines which states appear promising. A useful admissible heuristic can reduce unnecessary exploration while retaining the shortest-path guarantee. An aggressive heuristic such as `2 × Manhattan` can overestimate and therefore is not guaranteed to preserve optimality.

**4. What did the LLM contribute?**  
It helped translate the pre-specified design into Python, including the priority queue, heuristic functions, path reconstruction, BFS comparison, and test structure.

**5. What can go wrong without testing?**  
The program can contain incorrect state handling, invalid transitions, duplicate exploration, bad path reconstruction, an incorrect heuristic, or failure to terminate on impossible problems while still producing plausible-looking output.

The laboratory's final lesson is that the scientific idea is search, A* is an algorithmic approach, and the LLM is an engineering tool; the engineer remains responsible for understanding, testing, and validating the result.


In [2]:
# Compact reproduction of the recorded experimental results.
print('Original: BFS = 40 moves / 64 expansions; A* Manhattan = 40 moves / 64 expansions')
print('Trivial: BFS = 1 / 2; A* = 1 / 2')
print('No solution: BFS = failure / 9; A* = failure / 9')
print('Alternative paths: BFS = 4 / 10; A* = 4 / 5')
print('Heuristics: h=0 -> 40/64; Manhattan -> 40/64; Euclidean -> 40/64; 2*Manhattan -> 40/64')


Original: BFS = 40 moves / 64 expansions; A* Manhattan = 40 moves / 64 expansions
Trivial: BFS = 1 / 2; A* = 1 / 2
No solution: BFS = failure / 9; A* = failure / 9
Alternative paths: BFS = 4 / 10; A* = 4 / 5
Heuristics: h=0 -> 40/64; Manhattan -> 40/64; Euclidean -> 40/64; 2*Manhattan -> 40/64
